In [1]:
!uv pip install duckdb[all]

Using Python 3.11.0 environment at: C:\Users\vapormusic\miniconda3\envs\ass3
Audited 1 package in 1.05s


In [2]:
import sqlite3
import pandas as pd
import glob

In [4]:


con = sqlite3.connect("database_gov.db")

# Apply schema
with open("./Collected_by_the_Government/schema.sql") as f:
    con.executescript(f.read())

# Auto-import all CSVs (filename = table name)
for file in glob.glob("./Collected_by_the_Government/*.csv"):
    table = file.removesuffix(".csv").split("\\")[-1]
    print(f"Importing {file} into {table}...")
    df = pd.read_csv(file)
    df.to_sql(table, con, if_exists="append", index=False)

con.close()

con = sqlite3.connect("database_jour.db")

# Apply schema
with open("./Collected_by_the_Journalist/schema.sql") as f:
    con.executescript(f.read())

# Auto-import all CSVs (filename = table name)
for file in glob.glob("./Collected_by_the_Journalist/*.csv"):
    table = file.removesuffix(".csv").split("\\")[-1]
    print(f"Importing {file} into {table}...")
    df = pd.read_csv(file)
    df.to_sql(table, con, if_exists="append", index=False)

con.close()

OperationalError: table meetings already exists

In [5]:
import duckdb
duckdb.sql("""
INSTALL sqlite;
LOAD sqlite;
ATTACH 'database_jour.db' (TYPE sqlite);
USE database_jour;
ATTACH 'database_gov.db' (TYPE sqlite);
USE database_gov;
""")

In [6]:
duckdb.sql("""
    select * from database_jour.topics
""")

┌─────────────────────────┬─────────────────────────┬───────────────────────────────────────────────────────────────────────────┐
│        topic_id         │       short_topic       │                                long_topic                                 │
│         varchar         │         varchar         │                                  varchar                                  │
├─────────────────────────┼─────────────────────────┼───────────────────────────────────────────────────────────────────────────┤
│ expanding_tourist_wharf │ expanding_tourist_wharf │ Expanding the tourist wharf/infrastructure in Haacklee                    │
│ statue_john_smoth       │ statue_john_smoth       │ Statue to honor John Smoth in Port Grove                                  │
│ deep_fishing_dock       │ deep_fishing_dock       │ Deep maintenance to the commercial fishing dock at Himark                 │
│ new_crane_lomark        │ new_crane_lomark        │ Money for a new crane at Lomark     

In [42]:
nodes = duckdb.sql("""
    select b.topic_id, 
           a.discussion_id as target_id, 
           'discussion' as target_type,
           a.people_id,
           case when contains(industry, 'vessel') and contains(industry, 'tourism') then 0
                when contains(industry, 'vessel') then sentiment
                when contains(industry, 'tourism') then sentiment * -1
           else 0 end as sentiment,
           case when contains(industry, 'vessel') and contains(industry, 'tourism') then 'both'
                when contains(industry, 'vessel') then 'fishing'
                when contains(industry, 'tourism') then 'tourism'
           else 'other' end as industry
    from database_jour.discussion_people_participations a
    left join database_jour.discussion_topics  b 
    on a.discussion_id = b.discussion_id
    where  contains(industry, 'vessel') or contains(industry, 'tourism') 
    union all
    select b.topic_id, 
           a.plan_id as target_id, 
           'plan' as target_type,
           a.people_id,
           case when contains(industry, 'vessel') and contains(industry, 'tourism') then 0
                when contains(industry, 'vessel') then sentiment
                when contains(industry, 'tourism') then sentiment * -1           
           else 0 end as sentiment,
           case when contains(industry, 'vessel') and contains(industry, 'tourism') then 'both'
                when contains(industry, 'vessel') then 'fishing'
                when contains(industry, 'tourism') then 'tourism'
           else 'other' end as industry
    from database_jour.plan_people_participations a
    left join database_jour.plan_topics  b 
    on a.plan_id = b.plan_id
    where  contains(industry, 'vessel') or contains(industry, 'tourism') 

""").to_df()

nodes

,topic_id,target_id,target_type,people_id,sentiment,industry
0,expanding_tourist_wharf,expanding_tourist_wharf_Meeting_7_Initial_View...,discussion,Seal,-0.10,tourism
1,expanding_tourist_wharf,expanding_tourist_wharf_Travel_Harbor_Route_So...,discussion,Simone Kat,-0.50,tourism
2,expanding_tourist_wharf,expanding_tourist_wharf_Meeting_8_Expand_Ideas...,discussion,Seal,-0.10,tourism
3,expanding_tourist_wharf,expanding_tourist_wharf_Travel_Harbor_Route_So...,discussion,Simone Kat,-0.50,tourism
4,expanding_tourist_wharf,expanding_tourist_wharf_Meeting_9_Start_Report...,discussion,Seal,-0.10,tourism
...,...,...,...,...,...,...
113,seafood_festival,seafood_festival_Meeting_1_Feasibility,plan,Simone Kat,0.00,both
114,heritage_walking_tour,heritage_walking_tour_Meeting_7_Outline,plan,Simone Kat,-1.00,tourism
115,waterfront_market,waterfront_market_Meeting_7_Benefits_Feasibility,plan,Tante Titan,-0.25,tourism
116,concert,concert_Meeting_7_Concert_Issues,plan,Tante Titan,-0.50,tourism


In [43]:
nodes.loc[(pd.isna(nodes['topic_id'])) &(nodes['target_id'].str.startswith('waterfront_market')), 'topic'] = 'waterfront_market'

In [21]:
# Average sentiment of all persons
duckdb.sql("""
    select people_id, 
           avg(sentiment) as avg_sentiment
    from (
           select people_id,
           case when contains(industry, 'vessel') and contains(industry, 'tourism') then null
                when contains(industry, 'vessel') then sentiment
                when contains(industry, 'tourism') then sentiment * -1
                else null end as sentiment 
           from database_jour.discussion_people_participations
           union all
           select people_id,
           case when contains(industry, 'vessel') and contains(industry, 'tourism') then null  
                when contains(industry, 'vessel') then sentiment
                when contains(industry, 'tourism') then sentiment * -1
                else null end as sentiment
           from database_jour.plan_people_participations
    )         
    group by people_id
""")

┌─────────────────┬──────────────────────┐
│    people_id    │    avg_sentiment     │
│     varchar     │        double        │
├─────────────────┼──────────────────────┤
│ Teddy Goldstein │              0.71875 │
│ Seal            │ 0.009999999999999998 │
│ Ed Helpsford    │                 0.25 │
│ Carol Limpet    │  -0.6818181818181818 │
│ Simone Kat      │  -0.5299999999999999 │
│ Tante Titan     │ -0.47058823529411764 │
└─────────────────┴──────────────────────┘

In [3]:
trip_mapped= duckdb.sql("""
    with trip_mapped as (
    select * 
    from database_jour.trip_places a,
            database_jour.places b
        where a.place_id = b.place_id
    )
    select place_id, name, lat, lon, zone, zone_detail, count(*) as cnt
    from trip_mapped
    group by place_id, name,  lat, lon, zone, zone_detail
""").to_df()
trip_mapped.head()

,place_id,name,lat,lon,zone,zone_detail,cnt
0,South Paackland Ferry Terminal,South Paackland Ferry Terminal,-164.561771,39.085917,government,None,117
1,Paackland Ferry Terminal,Paackland Ferry Terminal,-164.408925,39.440779,government,None,84
2,Himark Ferry Terminal,Himark Ferry Terminal,-165.885612,39.655649,government,None,8
3,48580678,None,-165.604671,39.547945,commercial,None,16
4,48542802,None,-164.558539,39.087948,commercial,None,8


In [5]:
trip_mapped

,place_id,name,lat,lon,zone,zone_detail,cnt
0,Haacklee Ferry Terminal,Haacklee Ferry Terminal,-165.690525,38.986193,government,None,120
1,581717431,Tropics Environmental Hub,-164.561002,39.275134,government,environmental,3
2,30187146,None,-165.888367,39.666132,residential,None,2
3,48511187,None,-165.596258,39.543346,commercial,None,13
4,37882709,None,-164.337982,39.421994,commercial,None,1
...,...,...,...,...,...,...,...
154,36955939,None,-165.951764,39.095616,residential,None,1
155,48567071,None,-164.556739,39.081576,commercial,None,1
156,48498977,None,-165.593196,39.528220,residential,None,1
157,48543659,Harborfront Boutiques,-164.557744,39.087195,commercial,shops,1


In [6]:
!uv pip install nbformat>=4.2.0

Using Python 3.11.0 environment at: C:\Users\vapormusic\miniconda3\envs\ass3
Audited 1 package in 10ms


In [19]:
# Draw the map of the places lat and lon
# Color the points by red if "tourism", blue if "industrial", and gray if any other
# Size the points by the count of trips to that place
# Paint the oceanus_map.geojson as the background


import plotly.express as px
import plotly.graph_objects as go
import geopandas as gpd
import pandas as pd

# Load the oceanus map
oceanus_map = gpd.read_file("oceanus_map.geojson")

# Prepare trip_mapped as a DataFrame
df = trip_mapped.copy()

# # Filter place id  in Himark Ferry Terminal 30179401 30188848
# df = df[df["place_id"].isin(['30179401', '30188848', 'Himark Ferry Terminal'])]



df["color"] = df["zone"].apply(
    lambda x: "red" if x == "tourism" else ("blue" if x == "industrial" else "gray")
)
df["size"] = df["cnt"] * 10

# Build the interactive scatter map
fig = px.scatter_mapbox(
    df,
    lat="lon",         # your column named "lon" holds latitude
    lon="lat",         # your column named "lat" holds longitude
    color="zone",
    size="cnt",
    color_discrete_map={
        "tourism": "red",
        "industrial": "blue",
    },
    hover_data={"lat": True, "lon": True, "cnt": True, "zone": True, "place_id": True, "name": True},
    size_max=30,
    opacity=0.8,
    title="Trip Mapped Places",
    mapbox_style="carto-positron",
    zoom=9,
)

# Overlay the GeoJSON boundary
fig.update_layout(
    mapbox={
        "layers": [
            {
                "source": oceanus_map.__geo_interface__,
                "type": "fill",
                "color": "lightblue",
                "opacity": 0.4,
            },
            {
                "source": oceanus_map.__geo_interface__,
                "type": "line",
                "color": "black",
                "line": {"width": 1},
            },
        ]
    },
    margin={"r": 0, "t": 40, "l": 0, "b": 0},
    legend_title_text="Zone",
)

# Change layout size to 800x600, force the map to be centered and zoomed to show all points
fig.update_layout(
    width=800,
    height=600,
    mapbox=dict(
        center={"lat": df["lon"].mean(), "lon": df["lat"].mean()},
        zoom=7,
    ),
)

fig.show()

 



C:\Users\vapormusic\AppData\Local\Temp\ipykernel_45380\1436065740.py:29: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig = px.scatter_mapbox(


In [4]:
duckdb.sql("""
    with trip_mapped as (
        select * 
        from database_jour.trip_places a,
                database_jour.places b
            where a.place_id = b.place_id
    )
    select distinct name, zone, zone_detail
    from  trip_mapped
    where name is not null
""").to_df()

,name,zone,zone_detail
0,Centralia Checkpoint Office,government,customs
1,The Bait & Stich,tourism,restaurant
2,Anchor Way Grill,tourism,restaurant
3,Paackland Control Office,government,inspections
4,Paackland Admin Center,government,administrative office
5,Jordan Administrative Center,government,None
6,Suna Spit,connector,None
7,Tours Central Ticketing,tourism,trips
8,Lomark Civic Plaza,government,city hall
9,Dock & Roll Hall of Fame,tourism,restaurant


In [5]:
duckdb.sql("""
        create or replace table loc_travel as (
                select f.topic_id, f.long_topic, b.plan_id, b.long_title as plan_title, b.plan_type, d.*
                from    
                        database_jour.plans b,
                        database_jour.travel_links c,
                        database_jour.places d,
                        database_jour.plan_topics e,
                        database_jour.topics f
                where b.plan_id = c.plan_id and c.place_id = d.place_id
                and  b.plan_id = e.plan_id and e.topic_id = f.topic_id
        ) 
""")
loc_travel = duckdb.sql("""
        select *
        from loc_travel
""").to_df()
loc_travel

,topic_id,long_topic,plan_id,plan_title,plan_type,place_id,name,lat,lon,zone,zone_detail
0,expanding_tourist_wharf,Expanding the tourist wharf/infrastructure in ...,expanding_tourist_wharf_Travel_Harbor_Route_So...,Travel to Harbor Route Solutions in Haacklee,Travel,35889363,Harbor Route Solutions,-165.680418,38.989466,industrial,None
1,statue_john_smoth,Statue to honor John Smoth in Port Grove,statue_john_smoth_Travel_The_Bait_Stich,Travel to The Bait & Stich,Travel,2519420987,The Bait & Stich,-165.957039,39.094819,tourism,restaurant
2,deep_fishing_dock,Deep maintenance to the commercial fishing doc...,deep_fishing_dock_Travel_High_Seas_Fishing_Inc,Travel to High Seas Fishing Inc. in Himark,Travel,3753651731,High Seas Fishing Inc.,-165.886312,39.654079,industrial,None
3,new_crane_lomark,Money for a new crane at Lomark,new_crane_lomark_Travel_Blue_Wave_Shipping,Travel to Blue Wave Shipping in Lomark,Travel,491334376,Blue Wave Shipping,-165.594367,39.556995,industrial,None
4,fish_vacuum,Fish Vacuum,fish_vacuum_Travel_Bay_Harvest_Corporation,Travel to Bay Harvest Corporation in Himark,Travel,30192735,Bay Harvest Corporation,-165.884842,39.656382,industrial,None
5,low_volume_crane,Low-volume unload crane in Haacklee,low_volume_crane_Travel_Harbor_Route_Solutions,Travel to Harbor Route Solutions in Haacklee,Travel,35889363,Harbor Route Solutions,-165.680418,38.989466,industrial,None
6,low_volume_crane,Low-volume unload crane in Haacklee,low_volume_crane_Travel_Harbor_Builders,Travel to Harbor Builders in Haacklee,Travel,35922181,Harbor Builders,-165.671068,38.994728,industrial,None
7,low_volume_crane,Low-volume unload crane in Haacklee,low_volume_crane_Travel_Haacklee_Assembly_Co,Travel to Haacklee Assembly Co. in Haacklee,Travel,35872967,Haacklee Assembly Co.,-165.679911,38.991395,industrial,None
8,affordable_housing,Affordable housing for fishing workers (confli...,affordable_housing_Travel_Waveside_Townhomes,Travel to Waveside Townhomes in Lomark,Travel,48468290,Waveside Townhomes,-165.600677,39.525001,residential,None
9,affordable_housing,Affordable housing for fishing workers (confli...,affordable_housing_Travel_Tidewater_Flats,Travel to Tidewater Flats in Lomark,Travel,48480127,Tidewater Flats,-165.570668,39.539331,residential,None


In [6]:
time_trip_spend = duckdb.sql("""
    select a.*, b.*, c.*, d.people_id 
    from 
      database_jour.trips a,
     database_jour.trip_places b,
        database_jour.places c,
        database_jour.trip_people d
        where a.trip_id = b.trip_id and b.place_id = c.place_id and a.trip_id = d.trip_id
    order by a.trip_id, time
""").to_df()


# Create index column by trip_id and time
time_trip_spend["index"] = time_trip_spend.groupby("trip_id").cumcount()
time_trip_spend["index_lead"] = time_trip_spend.groupby("trip_id")["index"].shift(-1).fillna(9999).astype(int)

# Convert start_time, end_time, time to datetime
time_trip_spend["date"] = time_trip_spend["date"].str.replace("0040","2040")
time_trip_spend["start_time"] = pd.to_datetime(time_trip_spend["date"] + " " + time_trip_spend["start_time"])
time_trip_spend["end_time"] = pd.to_datetime(time_trip_spend["date"] + " " + time_trip_spend["end_time"])
time_trip_spend["time"] = pd.to_datetime(time_trip_spend["time"].str.replace("0040","2040"))
time_trip_spend["time_lead"] = time_trip_spend.groupby("trip_id")["time"].shift(-1).fillna(time_trip_spend["time"])

# Calculate the time spend in each place
# if index = 0 => time = start_time - time
# if index = index_lead - 1 => time = time_lead - time
# else time = end_time - time
def calculate_time(row):
    if row["index"] == 0:
        return row["time"] - row["start_time"]
    elif row["index"] == row["index_lead"] - 1:
        return row["time_lead"] - row["time"]
    else:
        if row["end_time"] - row["time"] < pd.Timedelta(0):
            return row["end_time"] + pd.Timedelta(days=1) - row["time"]
        else:
            return row["end_time"]  - row["time"]
time_trip_spend["time_spend"] = time_trip_spend.apply(calculate_time, axis=1)
time_trip_spend[["trip_id", "people_id", "place_id", "name", "time", "time_spend", "zone", "zone_detail"]]

,trip_id,people_id,place_id,name,time,time_spend,zone,zone_detail
0,trip_0,Simone Kat,582184557,Pacific Nature Bureau,2040-04-24 16:17:00,0 days 07:17:00,government,environmental
1,trip_0,Simone Kat,581853838,Jordan Administrative Center,2040-04-24 16:17:00,0 days 02:43:00,government,None
2,trip_0,Simone Kat,South Paackland Ferry Terminal,South Paackland Ferry Terminal,2040-04-24 19:00:00,0 days 02:00:00,government,None
3,trip_0,Simone Kat,Haacklee Ferry Terminal,Haacklee Ferry Terminal,2040-04-24 21:00:00,0 days 00:00:00,government,None
4,trip_1,Seal,37830690,None,2040-06-16 06:29:00,0 days 00:00:00,commercial,None
...,...,...,...,...,...,...,...,...
1358,trip_97,Simone Kat,Haacklee Ferry Terminal,Haacklee Ferry Terminal,2040-06-05 21:00:00,0 days 00:00:00,government,None
1359,trip_98,Tante Titan,Haacklee Ferry Terminal,Haacklee Ferry Terminal,2040-05-24 13:00:00,0 days 06:00:00,government,None
1360,trip_98,Tante Titan,South Paackland Ferry Terminal,South Paackland Ferry Terminal,2040-05-24 19:00:00,0 days 00:00:00,government,None
1361,trip_99,Seal,2564318352,Eastside Memorial Toll,2040-04-26 07:10:00,0 days 00:00:00,government,tolls


In [ ]:
import numpy as np
np.average([0.4, -0.2, -0.6, 0.7, -1])

np.float64(-0.13999999999999999)

: 

In [10]:
time_trip_spend.groupby("people_id").agg({"time_spend": "sum", "trip_id": "nunique"}).sort_values("time_spend", ascending=False)

,time_spend,trip_id
people_id,,
Carol Limpet,25 days 02:04:00,69
Simone Kat,17 days 23:31:00,67
Seal,16 days 10:34:00,53
Ed Helpsford,15 days 19:05:00,61
Teddy Goldstein,10 days 12:35:00,43
Tante Titan,4 days 03:30:12,49


In [10]:
loc_travel

,topic_id,long_topic,plan_id,plan_title,plan_type,place_id,name,lat,lon,zone,zone_detail
0,expanding_tourist_wharf,Expanding the tourist wharf/infrastructure in ...,expanding_tourist_wharf_Travel_Harbor_Route_So...,Travel to Harbor Route Solutions in Haacklee,Travel,35889363,Harbor Route Solutions,-165.680418,38.989466,industrial,None
1,statue_john_smoth,Statue to honor John Smoth in Port Grove,statue_john_smoth_Travel_The_Bait_Stich,Travel to The Bait & Stich,Travel,2519420987,The Bait & Stich,-165.957039,39.094819,tourism,restaurant
2,deep_fishing_dock,Deep maintenance to the commercial fishing doc...,deep_fishing_dock_Travel_High_Seas_Fishing_Inc,Travel to High Seas Fishing Inc. in Himark,Travel,3753651731,High Seas Fishing Inc.,-165.886312,39.654079,industrial,None
3,new_crane_lomark,Money for a new crane at Lomark,new_crane_lomark_Travel_Blue_Wave_Shipping,Travel to Blue Wave Shipping in Lomark,Travel,491334376,Blue Wave Shipping,-165.594367,39.556995,industrial,None
4,fish_vacuum,Fish Vacuum,fish_vacuum_Travel_Bay_Harvest_Corporation,Travel to Bay Harvest Corporation in Himark,Travel,30192735,Bay Harvest Corporation,-165.884842,39.656382,industrial,None
5,low_volume_crane,Low-volume unload crane in Haacklee,low_volume_crane_Travel_Harbor_Route_Solutions,Travel to Harbor Route Solutions in Haacklee,Travel,35889363,Harbor Route Solutions,-165.680418,38.989466,industrial,None
6,low_volume_crane,Low-volume unload crane in Haacklee,low_volume_crane_Travel_Harbor_Builders,Travel to Harbor Builders in Haacklee,Travel,35922181,Harbor Builders,-165.671068,38.994728,industrial,None
7,low_volume_crane,Low-volume unload crane in Haacklee,low_volume_crane_Travel_Haacklee_Assembly_Co,Travel to Haacklee Assembly Co. in Haacklee,Travel,35872967,Haacklee Assembly Co.,-165.679911,38.991395,industrial,None
8,affordable_housing,Affordable housing for fishing workers (confli...,affordable_housing_Travel_Waveside_Townhomes,Travel to Waveside Townhomes in Lomark,Travel,48468290,Waveside Townhomes,-165.600677,39.525001,residential,None
9,affordable_housing,Affordable housing for fishing workers (confli...,affordable_housing_Travel_Tidewater_Flats,Travel to Tidewater Flats in Lomark,Travel,48480127,Tidewater Flats,-165.570668,39.539331,residential,None


In [11]:
pd.merge(time_trip_spend, loc_travel[["place_id", "name", "zone", "zone_detail", "topic_id", "long_topic", "plan_id", "plan_title", "plan_type"]]
         , on=["place_id", "name", "zone", "zone_detail"], how="left").dropna(subset=["plan_id"])

,trip_id,date,start_time,end_time,trip_id_1,place_id,time,place_id_1,name,lat,...,people_id,index,index_lead,time_lead,time_spend,topic_id,long_topic,plan_id,plan_title,plan_type
12,trip_101,2040-07-06,2040-07-06 09:00:00,2040-07-06 21:00:00,trip_101,37828755,2040-07-06 15:05:00,37828755,Dock & Roll Hall of Fame,-164.358079,...,Simone Kat,0,1,2040-07-06 19:00:00,0 days 06:05:00,seafood_festival,Hosting an Annual Seafood Festival in Paackland,seafood_festival_Travel_Dock_Roll_Hall_of_Fame,Travel to Dock & Roll Hall of Fame in Paackland,Travel
35,trip_106,2040-04-19,2040-04-19 07:00:00,2040-04-19 19:00:00,trip_106,30188848,2040-04-19 11:26:00,30188848,Anchor Way Grill,-165.890243,...,Tante Titan,0,1,2040-04-19 11:57:00,0 days 04:26:00,renaming_park_himark,Renaming a park in Himark,renaming_park_himark_Travel_Anchor_Way_Grill,Discuss travel plans to Anchor Way Grill,Travel
38,trip_106,2040-04-19,2040-04-19 07:00:00,2040-04-19 19:00:00,trip_106,9196630064,2040-04-19 11:57:00,9196630064,Himark City Hall,-165.884104,...,Tante Titan,3,4,2040-04-19 12:00:00,0 days 00:03:00,renaming_park_himark,Renaming a park in Himark,renaming_park_himark_Travel_Himark_City_Hall,Travel to Himark City Hall,Travel
44,trip_108,2040-07-20,2040-07-20 09:31:00,2040-07-20 11:30:00,trip_108,37828755,2040-07-20 10:33:00,37828755,Dock & Roll Hall of Fame,-164.358079,...,Tante Titan,1,2,2040-07-20 11:05:00,0 days 00:32:00,seafood_festival,Hosting an Annual Seafood Festival in Paackland,seafood_festival_Travel_Dock_Roll_Hall_of_Fame,Travel to Dock & Roll Hall of Fame in Paackland,Travel
64,trip_110,2040-07-17,2040-07-17 09:00:00,2040-07-17 21:13:00,trip_110,48475647,2040-07-17 21:13:00,48475647,Lomark Civic Plaza,-165.597194,...,Carol Limpet,5,9999,2040-07-17 21:13:00,0 days 00:00:00,name_inspection_office,Putting a name on the inspection office in Lomark,name_inspection_office_Travel_Lomark_Civic_Plaza,Travel to Lomark Civic Plaza,Travel
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1339,trip_90,2040-07-09,2040-07-09 08:42:00,2040-07-09 09:28:00,trip_90,9196630064,2040-07-09 08:42:00,9196630064,Himark City Hall,-165.884104,...,Teddy Goldstein,0,1,2040-07-09 09:28:00,0 days 00:00:00,renaming_park_himark,Renaming a park in Himark,renaming_park_himark_Travel_Himark_City_Hall,Travel to Himark City Hall,Travel
1340,trip_90,2040-07-09,2040-07-09 08:42:00,2040-07-09 09:28:00,trip_90,30188848,2040-07-09 09:28:00,30188848,Anchor Way Grill,-165.890243,...,Teddy Goldstein,1,2,2040-07-09 09:28:00,0 days 00:00:00,renaming_park_himark,Renaming a park in Himark,renaming_park_himark_Travel_Anchor_Way_Grill,Discuss travel plans to Anchor Way Grill,Travel
1343,trip_91,2040-04-01,2040-04-01 07:57:00,2040-04-01 08:55:00,trip_91,30188848,2040-04-01 08:55:00,30188848,Anchor Way Grill,-165.890243,...,Teddy Goldstein,1,9999,2040-04-01 08:55:00,0 days 00:00:00,renaming_park_himark,Renaming a park in Himark,renaming_park_himark_Travel_Anchor_Way_Grill,Discuss travel plans to Anchor Way Grill,Travel
1352,trip_94,2040-07-15,2040-07-15 09:00:00,2040-07-15 21:00:00,trip_94,37828755,2040-07-15 15:50:00,37828755,Dock & Roll Hall of Fame,-164.358079,...,Simone Kat,1,2,2040-07-15 19:00:00,0 days 03:10:00,seafood_festival,Hosting an Annual Seafood Festival in Paackland,seafood_festival_Travel_Dock_Roll_Hall_of_Fame,Travel to Dock & Roll Hall of Fame in Paackland,Travel


## Check on the difference

### Trips

In [12]:
# Compare the different in trips between jouralist and government data
hidden_trips = duckdb.sql("""
    select * 
    from database_jour.trips a
    left join database_gov.trips b
    on a.trip_id = b.trip_id
    where b.trip_id is null
""").to_df()

hidden_trips

,trip_id,date,start_time,end_time,trip_id_1,date_1,start_time_1,end_time_1
0,trip_2,2040-06-28,08:23:00,08:23:00,None,None,None,None
1,trip_3,2040-06-06,07:28:00,07:59:00,None,None,None,None
2,trip_6,2040-05-15,08:44:00,10:21:00,None,None,None,None
3,trip_7,0040-07-17,09:31:00,03:00:00,None,None,None,None
4,trip_9,0040-04-24,08:56:00,11:40:00,None,None,None,None
...,...,...,...,...,...,...,...,...
143,trip_332,2040-05-06,08:44:00,11:10:00,None,None,None,None
144,trip_333,2040-04-27,06:59:00,06:59:00,None,None,None,None
145,trip_336,2040-06-23,08:05:00,09:35:00,None,None,None,None
146,trip_339,0040-07-03,07:47:00,11:10:00,None,None,None,None


In [13]:
time_trip_spend

,trip_id,date,start_time,end_time,trip_id_1,place_id,time,place_id_1,name,lat,lon,zone,zone_detail,people_id,index,index_lead,time_lead,time_spend
0,trip_0,2040-04-24,2040-04-24 09:00:00,2040-04-24 21:00:00,trip_0,582184557,2040-04-24 16:17:00,582184557,Pacific Nature Bureau,-164.563348,39.271829,government,environmental,Simone Kat,0,1,2040-04-24 16:17:00,0 days 07:17:00
1,trip_0,2040-04-24,2040-04-24 09:00:00,2040-04-24 21:00:00,trip_0,581853838,2040-04-24 16:17:00,581853838,Jordan Administrative Center,-164.561910,39.271379,government,None,Simone Kat,1,2,2040-04-24 19:00:00,0 days 02:43:00
2,trip_0,2040-04-24,2040-04-24 09:00:00,2040-04-24 21:00:00,trip_0,South Paackland Ferry Terminal,2040-04-24 19:00:00,South Paackland Ferry Terminal,South Paackland Ferry Terminal,-164.561771,39.085917,government,None,Simone Kat,2,3,2040-04-24 21:00:00,0 days 02:00:00
3,trip_0,2040-04-24,2040-04-24 09:00:00,2040-04-24 21:00:00,trip_0,Haacklee Ferry Terminal,2040-04-24 21:00:00,Haacklee Ferry Terminal,Haacklee Ferry Terminal,-165.690525,38.986193,government,None,Simone Kat,3,9999,2040-04-24 21:00:00,0 days 00:00:00
4,trip_1,2040-06-16,2040-06-16 06:29:00,2040-06-16 12:36:00,trip_1,37830690,2040-06-16 06:29:00,37830690,None,-164.384825,39.427188,commercial,None,Seal,0,1,2040-06-16 10:38:00,0 days 00:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1358,trip_97,2040-06-05,2040-06-05 09:00:00,2040-06-05 21:00:00,trip_97,Haacklee Ferry Terminal,2040-06-05 21:00:00,Haacklee Ferry Terminal,Haacklee Ferry Terminal,-165.690525,38.986193,government,None,Simone Kat,2,9999,2040-06-05 21:00:00,0 days 00:00:00
1359,trip_98,2040-05-24,2040-05-24 07:00:00,2040-05-24 19:00:00,trip_98,Haacklee Ferry Terminal,2040-05-24 13:00:00,Haacklee Ferry Terminal,Haacklee Ferry Terminal,-165.690525,38.986193,government,None,Tante Titan,0,1,2040-05-24 19:00:00,0 days 06:00:00
1360,trip_98,2040-05-24,2040-05-24 07:00:00,2040-05-24 19:00:00,trip_98,South Paackland Ferry Terminal,2040-05-24 19:00:00,South Paackland Ferry Terminal,South Paackland Ferry Terminal,-164.561771,39.085917,government,None,Tante Titan,1,9999,2040-05-24 19:00:00,0 days 00:00:00
1361,trip_99,2040-04-26,2040-04-26 07:10:00,2040-04-26 09:40:00,trip_99,2564318352,2040-04-26 07:10:00,2564318352,Eastside Memorial Toll,-164.349956,39.423983,government,tolls,Seal,0,1,2040-04-26 09:40:00,0 days 00:00:00


In [14]:
loc_travel[["place_id", "name", "zone", "zone_detail", "topic_id", "long_topic", "plan_id", "plan_title", "plan_type"]]

,place_id,name,zone,zone_detail,topic_id,long_topic,plan_id,plan_title,plan_type
0,35889363,Harbor Route Solutions,industrial,None,expanding_tourist_wharf,Expanding the tourist wharf/infrastructure in ...,expanding_tourist_wharf_Travel_Harbor_Route_So...,Travel to Harbor Route Solutions in Haacklee,Travel
1,2519420987,The Bait & Stich,tourism,restaurant,statue_john_smoth,Statue to honor John Smoth in Port Grove,statue_john_smoth_Travel_The_Bait_Stich,Travel to The Bait & Stich,Travel
2,3753651731,High Seas Fishing Inc.,industrial,None,deep_fishing_dock,Deep maintenance to the commercial fishing doc...,deep_fishing_dock_Travel_High_Seas_Fishing_Inc,Travel to High Seas Fishing Inc. in Himark,Travel
3,491334376,Blue Wave Shipping,industrial,None,new_crane_lomark,Money for a new crane at Lomark,new_crane_lomark_Travel_Blue_Wave_Shipping,Travel to Blue Wave Shipping in Lomark,Travel
4,30192735,Bay Harvest Corporation,industrial,None,fish_vacuum,Fish Vacuum,fish_vacuum_Travel_Bay_Harvest_Corporation,Travel to Bay Harvest Corporation in Himark,Travel
5,35889363,Harbor Route Solutions,industrial,None,low_volume_crane,Low-volume unload crane in Haacklee,low_volume_crane_Travel_Harbor_Route_Solutions,Travel to Harbor Route Solutions in Haacklee,Travel
6,35922181,Harbor Builders,industrial,None,low_volume_crane,Low-volume unload crane in Haacklee,low_volume_crane_Travel_Harbor_Builders,Travel to Harbor Builders in Haacklee,Travel
7,35872967,Haacklee Assembly Co.,industrial,None,low_volume_crane,Low-volume unload crane in Haacklee,low_volume_crane_Travel_Haacklee_Assembly_Co,Travel to Haacklee Assembly Co. in Haacklee,Travel
8,48468290,Waveside Townhomes,residential,None,affordable_housing,Affordable housing for fishing workers (confli...,affordable_housing_Travel_Waveside_Townhomes,Travel to Waveside Townhomes in Lomark,Travel
9,48480127,Tidewater Flats,residential,None,affordable_housing,Affordable housing for fishing workers (confli...,affordable_housing_Travel_Tidewater_Flats,Travel to Tidewater Flats in Lomark,Travel


In [ ]:
# Obscured trips: trips that are in the journalist data but not in the government data
related_trips = pd.merge(time_trip_spend, loc_travel[["place_id", "name", "zone", "zone_detail", "topic_id", "long_topic", "plan_id", "plan_title", "plan_type"]], on=["place_id", "name", "zone", "zone_detail"], how="left").dropna(subset=["plan_id"])

hidden_trips_mapped = related_trips.merge(hidden_trips[["trip_id"]], on="trip_id", how="inner")

hidden_trips_mapped.groupby("topic_id").agg(
    trip_count=("trip_id", "nunique"),
    total_time_spend=("time_spend", "sum")
)

# Most of the obscured trips are related to "tourism" or "vanity" topics, besides seafood festival which benefits both tourism and fishing.

,trip_count,total_time_spend
topic_id,,
name_harbor_area,5,0 days 00:57:00
name_inspection_office,5,0 days 03:03:00
renaming_park_himark,22,0 days 05:02:00
seafood_festival,13,0 days 15:48:00
statue_john_smoth,5,0 days 00:57:00


In [16]:
hidden_trips_mapped.sort_values("topic_id")[["trip_id","topic_id", "time_spend", "long_topic", "plan_title", "plan_type"]]

,trip_id,topic_id,time_spend,long_topic,plan_title,plan_type
49,trip_88,name_harbor_area,0 days 00:00:00,Putting a name on pictureque harbor area in Po...,Travel to The Bait & Stich,Travel
19,trip_220,name_harbor_area,0 days 00:57:00,Putting a name on pictureque harbor area in Po...,Travel to The Bait & Stich,Travel
29,trip_293,name_harbor_area,0 days 00:00:00,Putting a name on pictureque harbor area in Po...,Travel to The Bait & Stich,Travel
33,trip_3,name_harbor_area,0 days 00:00:00,Putting a name on pictureque harbor area in Po...,Travel to The Bait & Stich,Travel
40,trip_37,name_harbor_area,0 days 00:00:00,Putting a name on pictureque harbor area in Po...,Travel to The Bait & Stich,Travel
24,trip_254,name_inspection_office,0 days 00:00:00,Putting a name on the inspection office in Lomark,Travel to Lomark Civic Plaza,Travel
45,trip_80,name_inspection_office,0 days 00:54:00,Putting a name on the inspection office in Lomark,Travel to Lomark Civic Plaza,Travel
10,trip_172,name_inspection_office,0 days 00:00:00,Putting a name on the inspection office in Lomark,Travel to Lomark Civic Plaza,Travel
15,trip_202,name_inspection_office,0 days 00:29:00,Putting a name on the inspection office in Lomark,Travel to Lomark Civic Plaza,Travel
35,trip_332,name_inspection_office,0 days 01:40:00,Putting a name on the inspection office in Lomark,Travel to Lomark Civic Plaza,Travel


### Plans

In [17]:
# Compare the different in plans between jouralist and government data
duckdb.sql("""
    select a.*, c.people_id
    from database_jour.plans a
    left join database_jour.plan_people_participations c
    on a.plan_id = c.plan_id
    left join database_gov.plans b
    on a.plan_id = b.plan_id
    where b.plan_id is null
""").to_df()

# Most of the obscured plans are related to "tourism" topics, besides seafood festival which benefits both tourism and fishing.
# All of them is by Tante Titan

,plan_id,short_title,long_title,plan_type,people_id
0,statue_john_smoth_Meeting_9_Feedback,statue_john_smoth_Meeting_9_Feedback,Gather community feedback on the proposed statue,feedback,Tante Titan
1,statue_john_smoth_Meeting_10_Presentation,statue_john_smoth_Meeting_10_Presentation,Provide presentation on feedback and costs,presentation,Tante Titan
2,renaming_park_himark_Travel_Anchor_Way_Grill,renaming_park_himark_Travel_Anchor_Way_Grill,Discuss travel plans to Anchor Way Grill,Travel,Tante Titan
3,renaming_park_himark_Travel_Sailors_Perch_Light,renaming_park_himark_Travel_Sailors_Perch_Light,Discuss travel plans to Sailor's Perch Light,Travel,Tante Titan
4,renaming_park_himark_Meeting_8_Compile_Nominat...,renaming_park_himark_Meeting_8_Compile_Nominat...,Compile nominations and conduct a community vote,proposal,Tante Titan
5,renaming_park_himark_Meeting_7_Announce_Name,renaming_park_himark_Meeting_7_Announce_Name,Announce the new park name and organize an unv...,take action,Tante Titan
6,renaming_park_himark_Meeting_7_Unveiling_Event,renaming_park_himark_Meeting_7_Unveiling_Event,Presentation on unveiling event,presentation,Tante Titan
7,name_harbor_area_Travel_The_Bait_Stich,name_harbor_area_Travel_The_Bait_Stich,Travel to The Bait & Stich,Travel,Tante Titan
8,name_harbor_area_Meeting_11_Draft_Names,name_harbor_area_Meeting_11_Draft_Names,Draft potential names for the customs office,proposal,Tante Titan
9,name_harbor_area_Travel_Harbor_Odyssey_Tours,name_harbor_area_Travel_Harbor_Odyssey_Tours,Travel to Harbor Odyssey Tours,Travel,Tante Titan


### Discussions

In [18]:
# Compare the different in plans between jouralist and government data
# All of them is by Tante Titan, bar for 2 with unknown people_id
duckdb.sql("""
    select a.*, c.people_id
    from database_jour.discussions a    
    left join database_jour.discussion_people_participations c
    on a.discussion_id = c.discussion_id
    left join database_gov.discussions b
    on a.discussion_id = b.discussion_id 
    where b.discussion_id is null
""").to_df()

,discussion_id,short_title,long_title,people_id
0,statue_john_smoth_Meeting_9_Gather_Feedback_Di...,statue_john_smoth_Meeting_9_Gather_Feedback_Di...,Gather community feedback on the proposed statue,Tante Titan
1,statue_john_smoth_Travel_The_Bait_Stich_Comple...,statue_john_smoth_Travel_The_Bait_Stich_Comple...,Discuss the completed travel plans to The Bait...,Tante Titan
2,statue_john_smoth_Meeting_10_Presentation_Disc...,statue_john_smoth_Meeting_10_Presentation_Disc...,Provide presentation on feedback and costs,Tante Titan
3,renaming_park_himark_Travel_Anchor_Way_Grill_D...,renaming_park_himark_Travel_Anchor_Way_Grill_D...,Discuss travel plans to Anchor Way Grill,Tante Titan
4,renaming_park_himark_Travel_Sailors_Perch_Ligh...,renaming_park_himark_Travel_Sailors_Perch_Ligh...,Discuss travel plans to Sailor's Perch Light,Tante Titan
5,renaming_park_himark_Travel_Anchor_Way_Grill_C...,renaming_park_himark_Travel_Anchor_Way_Grill_C...,Discuss the completed travel plans to Anchor W...,Tante Titan
6,renaming_park_himark_Travel_Sailors_Perch_Ligh...,renaming_park_himark_Travel_Sailors_Perch_Ligh...,Discuss the completed travel plans to Sailor's...,Tante Titan
7,renaming_park_himark_Meeting_8_Compile_Nominat...,renaming_park_himark_Meeting_8_Compile_Nominat...,Compile nominations and conduct a community vote,Tante Titan
8,renaming_park_himark_Meeting_7_Announce_Name_D...,renaming_park_himark_Meeting_7_Announce_Name_D...,Announce the new park name and organize an unv...,Tante Titan
9,renaming_park_himark_Meeting_7_Unveiling_Event...,renaming_park_himark_Meeting_7_Unveiling_Event...,Presentation on unveiling event,Tante Titan


In [20]:
# Compare the different in plans between jouralist and government data
# All of them is by Tante Titan, bar for 2 with unknown people_id
duckdb.sql("""
    select a.*
    from database_jour.discussion_people_participations a    
    left join database_gov.discussion_people_participations b
    on a.discussion_id = b.discussion_id 
    where b.discussion_id is null
""").to_df()

,discussion_id,people_id,sentiment,reason,industry
0,statue_john_smoth_Meeting_8_Proposal_Discussion,Tante Titan,1.00,Loves ceremonial gestures and believes they ca...,[]
1,statue_john_smoth_Meeting_9_Gather_Feedback_Di...,Tante Titan,1.00,Loves ceremonial gestures and believes they ca...,[]
2,statue_john_smoth_Travel_The_Bait_Stich_Comple...,Tante Titan,1.00,Loves ceremonial gestures and believes they ca...,[]
3,statue_john_smoth_Meeting_10_Presentation_Disc...,Tante Titan,1.00,Loves ceremonial gestures and believes they ca...,[]
4,renaming_park_himark_Meeting_4_Name_Ideas_Disc...,Tante Titan,1.00,Loves ceremonial gestures and believes they ca...,[]
5,renaming_park_himark_Travel_Anchor_Way_Grill_D...,Tante Titan,1.00,Loves ceremonial gestures and believes they ca...,[]
6,renaming_park_himark_Travel_Sailors_Perch_Ligh...,Tante Titan,1.00,Loves ceremonial gestures and believes they ca...,[]
7,renaming_park_himark_Travel_Anchor_Way_Grill_C...,Tante Titan,1.00,Loves ceremonial gestures and believes they ca...,[]
8,renaming_park_himark_Travel_Sailors_Perch_Ligh...,Tante Titan,1.00,Loves ceremonial gestures and believes they ca...,[]
9,renaming_park_himark_Meeting_8_Compile_Nominat...,Tante Titan,1.00,Loves ceremonial gestures and believes they ca...,[]


In [22]:

duckdb.sql("""
    select a.*
    from database_gov.discussion_people_participations a    
    where people_id == 'Tante Titan'
""").to_df()

,discussion_id,people_id,sentiment,reason,industry


In [23]:

duckdb.sql("""
    select a.*
    from database_gov.trip_people a    
    where people_id == 'Tante Titan'
""").to_df()

,trip_id,people_id,time
0,trip_33,Tante Titan,None
1,trip_227,Tante Titan,None
2,trip_106,Tante Titan,None
3,trip_178,Tante Titan,None


In [24]:

duckdb.sql("""
    select a.*
    from database_jour.trip_people a    
    where people_id == 'Tante Titan'
""").to_df()

,trip_id,people_id,time
0,trip_9,Tante Titan,None
1,trip_26,Tante Titan,None
2,trip_32,Tante Titan,None
3,trip_33,Tante Titan,None
4,trip_36,Tante Titan,None
5,trip_42,Tante Titan,None
6,trip_46,Tante Titan,None
7,trip_55,Tante Titan,None
8,trip_68,Tante Titan,None
9,trip_84,Tante Titan,None
